# PTB-XL — download, convert, label, visualize

[PTB-XL](https://physionet.org/content/ptb-xl/1.0.3/) (~21 799 × 10 s, 12-lead). Always subset — the full 500 Hz dump is ~3 GB.

Already-processed records are skipped. New waveforms are downloaded temporarily, converted, then the WFDB files are deleted.

In [1]:
from pathlib import Path
import sys

REPO = Path.cwd() if (Path.cwd() / "evaluation" / "common.py").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "evaluation"))

import pandas as pd
from IPython.display import display

import common as C

# --- subset ---
N_RECORDS = 24
SAMPLING = 500
STRAT_FOLDS = [10]
SUPERCLASSES = None
SEX = None
BALANCED_NORM_ABNORMAL = True
OVERWRITE_PROCESSED = False
CONSUME_WFDB = True
SEED = 42

RAW_DIR, PROC_DIR = C.dataset_dirs("ptb_xl")
print("raw:", RAW_DIR)
print("processed:", PROC_DIR)


raw: C:\Users\staso\OneDrive\Pulpit\Nauka\University\Thesis\code\ECG_Delineation\WTdelineator\data\evaluation\raw\ptb_xl
processed: C:\Users\staso\OneDrive\Pulpit\Nauka\University\Thesis\code\ECG_Delineation\WTdelineator\data\evaluation\processed\ptb_xl


## 1. Metadata only — then choose a subset

In [ ]:
meta, statements = C.load_ptbxl_catalogue()
print(f"PTB-XL rows: {len(meta)}  statements: {len(statements)}")

selected = C.select_ptbxl_subset(
    meta,
    statements,
    n_records=N_RECORDS,
    superclasses=SUPERCLASSES,
    strat_folds=STRAT_FOLDS,
    sex=SEX,
    balanced_norm_abnormal=BALANCED_NORM_ABNORMAL,
    seed=SEED,
)
print(
    selected.assign(supers=selected["diagnostic_superclasses"].map(lambda x: ",".join(x)))
    [["ecg_id", "patient_id", "sex", "age", "strat_fold", "is_norm", "supers", "filename_hr"]].head(12)
)
print("NORM / abnormal:", int(selected["is_norm"].sum()), "/", int((~selected["is_norm"]).sum()))


## 2. Convert (skip processed; temp download if needed)

In [ ]:
col = "filename_hr" if SAMPLING == 500 else "filename_lr"
results = []
for _, row in selected.iterrows():
    rec_id = f"{int(row['ecg_id']):05d}_{'hr' if SAMPLING == 500 else 'lr'}"
    results.append(
        C.acquire_and_convert_ptbxl(
            rec_id,
            row,
            statements,
            filename_hr=str(row[col]),
            overwrite=OVERWRITE_PROCESSED,
            consume=CONSUME_WFDB,
        )
    )
print(C.summarize_acquire(results))
failed = [r for r in results if r["status"] == "failed"]
if failed:
    print("failed:", failed[:10])

index = C.load_index(PROC_DIR)
display(index.head())
print(f"index rows: {len(index)}")


## 3. Visualize

In [2]:
index = C.load_index(PROC_DIR)
C.plot_category_counts(
    index["superclasses"].fillna("none").value_counts(),
    title="PTB-XL subset — diagnostic superclasses",
    color="#6a51a3",
)
is_norm = index["is_norm"].astype(str).str.lower().isin(["true", "1"])
_ = C.plot_count_panels(
    [
        (is_norm.map({True: "NORM", False: "abnormal"}).value_counts(), "NORM vs abnormal", "#31a354"),
        (index["sex"].fillna("unknown").value_counts(), "Sex", "#3182bd"),
    ]
)


In [ ]:
index = C.load_index(PROC_DIR)
is_norm = index["is_norm"].astype(str).str.lower().isin(["true", "1"])
norm_ids = index.loc[is_norm, "record_id"]
abn_ids = index.loc[~is_norm, "record_id"]
if len(norm_ids):
    print(f"NORM example: {norm_ids.iloc[0]}")
    _ = C.plot_ptbxl_record(str(norm_ids.iloc[0]), PROC_DIR)
if len(abn_ids):
    print(f"abnormal example: {abn_ids.iloc[0]}")
    _ = C.plot_ptbxl_record(str(abn_ids.iloc[0]), PROC_DIR)


NORM example: 01315_hr


abnormal example: 00400_hr
